# PDB Fetch Tools Examples

This notebook demonstrates the two PDB fetch tools:

- **`run_pdb_fetch_entry`** -- Fetch title, experimental method, and resolution
- **`run_pdb_fetch_fasta`** -- Retrieve chain sequences with protein/nucleotide classification

## Imports

In [1]:
from proto_tools import (
    PdbFetchConfig,
    PdbFetchEntryInput,
    PdbFetchFastaInput,
    run_pdb_fetch_entry,
    run_pdb_fetch_fasta,
)


## API Reference

### `PdbFetchEntryInput`

| Field | Type | Default | Description |
|---|---|---|---|
| `pdb_id` | `str` | *(required)* | PDB accession (e.g. `1LBG`) |

### `PdbFetchFastaInput`

| Field | Type | Default | Description |
|---|---|---|---|
| `pdb_id` | `str` | *(required)* | PDB accession (e.g. `1LBG`) |

### `PdbFetchConfig` (shared)

| Field | Type | Default | Description |
|---|---|---|---|
| `request_timeout_seconds` | `int` | `15` | HTTP timeout per request |
| `http_retries` | `int` | `3` | Max HTTP retries |
| `backoff_seconds` | `float` | `1.0` | Seconds to wait between retries (doubles after each attempt) |
| `user_agent` | `str` | `"proto-tools/pdb-fetch-v1"` | Identifier string sent to database APIs with each request |

### Output Types

**`PdbFetchEntryOutput`** — `title`, `method`, `resolution`, `source_url`

**`PdbFetchFastaOutput`** — `chains: List[PdbChain]`, `source_url`

**`PdbChain`** — `chain_id`, `header`, `sequence`, `is_protein`

## 1. Fetch Entry Metadata

Retrieve structure title, experimental method, and resolution for the lac repressor structure.

In [2]:
inputs = PdbFetchEntryInput(pdb_id="1LBG")

output = run_pdb_fetch_entry(inputs, PdbFetchConfig())

print(f"Title: {output.title}")
print(f"Method: {output.method}")
print(f"Resolution: {output.resolution} A")
print(f"Source: {output.source_url}")

Title: LACTOSE OPERON REPRESSOR BOUND TO 21-BASE PAIR SYMMETRIC OPERATOR DNA, ALPHA CARBONS ONLY
Method: X-RAY DIFFRACTION
Resolution: 4.8 A
Source: https://data.rcsb.org/rest/v1/core/entry/1LBG


## 2. Fetch FASTA Chains

Retrieve all chain sequences from a PDB entry with automatic protein/nucleotide classification.

In [3]:
inputs = PdbFetchFastaInput(pdb_id="1LBG")

output = run_pdb_fetch_fasta(inputs, PdbFetchConfig())

print(f"Chains: {len(output.chains)}")
print()

for chain in output.chains:
    chain_type = "protein" if chain.is_protein else "nucleotide"
    preview = chain.sequence[:50] + ("..." if len(chain.sequence) > 50 else "")
    print(f"  Chain {chain.chain_id}: {chain_type}, length={len(chain.sequence)}")
    print(f"    {preview}")

Chains: 2

  Chain 1: nucleotide, length=21
    GAATTGTGAGCGCTCACAATT
  Chain 2: protein, length=360
    MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELNYIPN...


## Export Results

Save fetched results to JSON format.

In [4]:
# Export FASTA chains to JSON
output.export("pdb_fasta_results", export_path="./example_output", file_format="json")
print("Exported to ./example_output/pdb_fasta_results.json")

Exported to ./example_output/pdb_fasta_results.json
